Sizing the 5-transistor OTA example with Bayesian Optimization (Ax) as a constraint satisfaction problem.

# Pre-body

## Clearing past runs (optional)

In [2]:
!rm -rf logs/ # clear logs
!rm -rf spice_out/

## IIC-OSIC Env Setup

In [3]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [4]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
# from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.designer_tools              import Project_Setup
from symxplorer.optimization.bayesian_ax    import Ax_Spice_Constraint_Satisfaction
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

# Instantiations


## Loading the project config

In [5]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

06:05:25 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
06:05:25 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/logs/SymXplorer_2025-10-14_06-05-25.log
06:05:25 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
06:05:25 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: TwoPointsDE, type=nevergrad, budget=10, random_seed=48
06:05:25 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
06:05:25 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
06:05:25 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
06:05:25 - SymXplorer.domains: [INFO] 	Number of target specs: 3
06:05:25 - SymXplorer.domains: [INFO] 		- TargetSpec(name=ugf, target=200e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=100.0, enable=True, description=Unitiy gain frequency)
06:05

Project_Setup(name='5T-OTA', description='5 Transistor OTA example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2'), netlist=PosixPath('spice/ota-5t_tb-loopgain.spice'), outdir=PosixPath('sizing/spice_out'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07)}), pvt=PVT(temp=25, corner='tt', supply=1.8), dut_params=[Param(name='x_dut_nfet_input_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False), Param(name='x_dut_nfet_input_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=N

## Create a SPICE simulator wrapper

In [6]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

06:05:25 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out
06:05:25 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:05:25 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
06:05:25 - SymXplorer.spicelib: [INFO] 	📝 Project: 5T-OTA
06:05:25 - SymXplorer.spicelib: [INFO] 	📜 Schematic: ota-5t_tb-loopgain
06:05:25 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out
06:05:25 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:05:25 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
06:05:25 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
06:05:25 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['v_dd', 'GND', 'v_ss', 'v_in', 'v_ena', 'vr1', 'net1', 'vf1', 'net2', 'net3', 'net4', 'net5', 'ne

## Create an optimizer object

In [7]:
circuit_optimizer = Ax_Spice_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

06:05:25 - SymXplorer.base_optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs
06:05:25 - SymXplorer.Ax: [INFO] started the <class 'symxplorer.optimization.bayesian_ax.Ax_Spice_Constraint_Satisfaction'> optimizer class


## Sanity Check

In [8]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

06:05:25 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
06:05:25 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
06:05:26 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out/sanity_check/5T-OTA_sanity.log
06:05:26 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out/sanity_check/5T-OTA_sanity.raw
06:05:26 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
06:05:26 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Main Body

## Optimization

In [9]:
circuit_optimizer.parameterize()

[RangeParameterConfig(name='x_dut_nfet_input_w', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_nfet_input_l', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_nfet_mirror_w', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_nfet_mirror_l', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_pfet_load_w', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_pfet_load_l', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None)]

In [10]:
_ = circuit_optimizer.optimize()

06:05:26 - SymXplorer.base_optimizer: [INFO] Optimization process started.
Optimizing:   0%|          | 0/10 [00:00<?, ?trial/s][INFO 10-14 06:05:26] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol'), GenerationNode(node_name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, model_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')]), GenerationNode(node_name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, model_key_override=None)], transition_criteria=[])]) chosen based on user input and problem structure.
[INFO 10-14 06:05:26] ax.api.client: Generated new trial 0 with parameters {'x_dut_nfet_input_w': 50.0, 'x_dut_nfet_input_l': 50.0, 'x_dut_nfet_mirror_w': 50.0, 'x_dut_nfet_mirror_l': 50.0, 'x_dut_pfet_load_w': 50.0, 'x_dut_pfet_load_l': 50.0}using GenerationNode CenterOfSearchSpace.
[INFO 10-14 06:05:26] ax.api.client: Trial 0 marked COMPL

In [11]:
circuit_optimizer.plot_score(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

06:05:39 - SymXplorer.plotter: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/loss_curve.html
06:05:39 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [12]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
# metadata

06:05:40 - SymXplorer.base_optimizer: [INFO] best score: -3.7234436043412034


In [13]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

x_dut_nfet_input_w: 1.80e-07
x_dut_nfet_input_l: 1.80e-07
x_dut_nfet_mirror_w: 1.80e-07
x_dut_nfet_mirror_l: 1.80e-07
x_dut_pfet_load_w: 1.80e-07
x_dut_pfet_load_l: 1.80e-07


In [14]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="out")

06:05:41 - SymXplorer.base_optimizer: [INFO] total score: -34.25122093405042
06:05:41 - SymXplorer.base_optimizer: [INFO] 	Spec 'ugf': curr_val=126434200.00000001, score=-30.754257431037658
06:05:41 - SymXplorer.base_optimizer: [INFO] 	Spec 'dcgain': curr_val=23.00322, score=-3.496963503012762
06:05:41 - SymXplorer.base_optimizer: [INFO] 	Spec 'pm': curr_val=87.0, score=0.0


### (3) Metric Trace

In [15]:
_ = circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='ugf', show=True)

06:05:41 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


In [16]:
circuit_optimizer.plot_score_value_by_spec(spec_name="dcgain", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="ugf", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="pm", show=True)

06:05:41 - SymXplorer.plotter: [INFO] 	min score -3.657572675771892; max score 0.0
06:05:41 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


06:05:41 - SymXplorer.plotter: [INFO] 	min score -68.05783628084394; max score -0.40014786427944316
06:05:41 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


06:05:41 - SymXplorer.plotter: [INFO] 	min score -16.514041292462945; max score 0.0
06:05:41 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


### (4) Design Space Exploration

In [17]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_pfet_load_w", param_y="x_dut_pfet_load_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_input_w", param_y="x_dut_nfet_input_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_mirror_w", param_y="x_dut_nfet_mirror_l", show=True)

06:05:41 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


06:05:41 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


06:05:41 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


(tensor([1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07,
         1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07]),
 tensor([1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07,
         1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07]))

# Ax-specific Visualiztion

In [ ]:
circuit_optimizer.optimizer.summarize() # Would output normalized values on the

,trial_index,arm_name,trial_status,generation_node,score,x_dut_nfet_input_w,x_dut_nfet_input_l,x_dut_nfet_mirror_w,x_dut_nfet_mirror_l,x_dut_pfet_load_w,x_dut_pfet_load_l
0,0,0_0,COMPLETED,CenterOfSearchSpace,-73.220635,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
1,1,1_0,COMPLETED,Sobol,-47.605876,32.518561,30.708895,34.998952,61.749450,21.467661,7.128493
2,2,2_0,COMPLETED,Sobol,-79.668547,61.774180,63.300315,52.707864,47.085739,75.002660,55.816044
3,3,3_0,COMPLETED,Sobol,-54.843150,90.539645,10.787816,17.404494,81.868132,62.333470,87.939423
4,4,4_0,COMPLETED,Sobol,-73.865730,16.728219,96.504166,95.301035,21.501326,41.171893,49.126971
5,5,5_0,COMPLETED,MBM,-10.394951,11.803402,0.000000,16.234286,79.662538,0.000000,6.960465
6,6,6_0,COMPLETED,MBM,-70.500780,0.000000,0.000000,0.000000,100.000000,0.000000,0.000000
7,7,7_0,COMPLETED,MBM,-68.654755,0.000000,0.000000,0.000000,75.447181,0.000000,12.677752
8,8,8_0,COMPLETED,MBM,-3.723444,5.980269,0.000000,20.454134,60.298790,0.000000,7.463016
9,9,9_0,COMPLETED,MBM,-62.897523,0.000000,0.000000,21.117674,72.253503,0.000000,69.301467


# Checkpointing

In [19]:
name = str(PROJECT_SETUP.ws_root/Path("sizing/data/checkpoint"))
circuit_optimizer.save_checkpoint(name=name)

06:05:41 - SymXplorer.base_optimizer: [INFO] ✅ Checkpoint saved to /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/data/checkpoint_2025-10-14_06-05-41.json
